In [ ]:
# Source - https://stackoverflow.com/a/77503061
# Posted by Rajnish Dubey
# Retrieved 2026-05-07, License - CC BY-SA 4.0

# import os, gzip, shutil
# dir_name = r'../data'
# def gz_extract(directory):
#     extension = ".gz"
#     os.chdir(directory)
#     for item in os.listdir(directory): # loop through items in dir
#       if item.endswith(extension): # check for ".gz" extension
#           gz_name = os.path.abspath(item) # get full path of files
#           file_name = (os.path.basename(gz_name)).rsplit('.',1)[0] #get file name for file within
#           with gzip.open(gz_name,"rb") as f_in, open(file_name,"wb") as f_out:
#               shutil.copyfileobj(f_in, f_out)
#           os.remove(gz_name) # delete zipped file      
# gz_extract(dir_name)


In [2]:
import re
def extract_gtf_attribute(series, key):
    return series.str.extract(fr'{key} "([^"]+)"', expand=False)

def get_appris_rank(attr):
    match = re.search(r'tag "appris_principal_(\d)"', attr)
    if match: return int(match.group(1))
    if 'tag "appris_principal"' in attr: return 1
    return 10  # Lowest priority

def format_interval(row):
    return f"{row['seqname']}:{row['start']}-{row['end']}({row['strand']})"

In [3]:
import pandas as pd
from pathlib import Path

gtf_path = Path('..') / 'data' / 'gencode.v49.primary_assembly.annotation.gtf'
rank_path = Path('..') / 'data' / 'gencode.v49.transcript_rankings.txt'

df = pd.read_csv(gtf_path, sep='\t', comment='#', header=None,
                 names=['seqname','source','feature','start','end','score','strand','frame','attribute'])

len(df)

7762316

In [4]:
# helpers
def parse_intervals(interval_string):
    if pd.isna(interval_string) or interval_string == '':
        return []
    parsed = []
    for interval in interval_string.split(';'):
        interval = interval.strip()
        if not interval:
            continue
        chrom, rest = interval.split(':', 1)
        pos, strand = rest.split('(')
        start_s, end_s = pos.split('-')
        parsed.append((chrom, int(start_s), int(end_s), strand.rstrip(')')))
    return parsed


def merge_interval_list(intervals):
    """Merge overlapping/adjacent genomic intervals (same chrom and strand)."""
    if not intervals:
        return []

    intervals = sorted(intervals, key=lambda x: (x[0], x[3], x[1], x[2]))
    merged = []
    current = list(intervals[0])

    for chrom, start, end, strand in intervals[1:]:
        same_locus = chrom == current[0] and strand == current[3]
        if same_locus and start <= current[2] + 1:
            current[2] = max(current[2], end)
        else:
            merged.append(tuple(current))
            current = [chrom, start, end, strand]

    merged.append(tuple(current))
    return merged


def sort_intervals_in_transcript_direction(intervals):
    """Sort intervals in 5'->3' transcript order (not genomic order)."""
    if not intervals:
        return []
    strand = intervals[0][3]
    return sorted(intervals, key=lambda x: x[1], reverse=(strand == '-'))


def intervals_to_string(intervals):
    return '; '.join(f'{chrom}:{start}-{end}({strand})' for chrom, start, end, strand in intervals)


def merge_cds_with_codons_start_cds_stop(row):
    cds_intervals = parse_intervals(row.get('CDS', ''))
    start_intervals = parse_intervals(row.get('start_codon', ''))
    stop_intervals = parse_intervals(row.get('stop_codon', ''))

    merged = merge_interval_list(cds_intervals + start_intervals + stop_intervals)
    merged = sort_intervals_in_transcript_direction(merged)
    return intervals_to_string(merged)

def classify_utr(row, codon_map):
    tid = row['transcript_id']
    if row['feature'] == 'CDS':
        return 'CDS'
    if tid not in codon_map.index:
        return 'UTR_Incomplete'

    try:
        start_c_start = codon_map.loc[tid, ('start', 'start_codon')]
        stop_c_end = codon_map.loc[tid, ('end', 'stop_codon')]
    except KeyError:
        return 'UTR_Unknown'

    if row['strand'] == '+':
        if row['end'] <= start_c_start:
            return "5' UTR"
        if row['start'] >= stop_c_end:
            return "3' UTR"
    else:
        # Invert for - strand
        if row['start'] >= start_c_start:
            return "5' UTR"
        if row['end'] <= stop_c_end:
            return "3' UTR"
    return 'UTR_Internal'


In [5]:
df['gene_name'] = extract_gtf_attribute(df['attribute'], 'gene_name')
df['gene_id'] = extract_gtf_attribute(df['attribute'], 'gene_id')
df['protein_id'] = extract_gtf_attribute(df['attribute'], 'protein_id')
df['transcript_id'] = extract_gtf_attribute(df['attribute'], 'transcript_id')
df['unfinished_mRNA'] = extract_gtf_attribute(df['attribute'], 'tag') == "mRNA_end_NF"
df['unfinished_CDS'] = extract_gtf_attribute(df['attribute'], 'tag') == "cds_end_NF"
df['transcript_type'] = extract_gtf_attribute(df['attribute'], 'transcript_type')

# Filter for ENST and Rank 1
ranks = pd.read_csv(rank_path, sep='\t', header=None, usecols=[1,3,4], names=['gene_id','rank','transcript_id'])
rank1_ensts = ranks.loc[(ranks['transcript_id'].str.startswith('ENST')) & (ranks['rank'] == 1), 'transcript_id'].unique()

# Select one transcript per gene based on APPRIS
transcripts = df[df['feature'] == 'transcript'].copy()
transcripts = transcripts[transcripts['transcript_id'].isin(rank1_ensts)].copy()
transcripts['appris_rank'] = transcripts['attribute'].apply(get_appris_rank)

final_best_ids = (
    transcripts.sort_values(['gene_id', 'appris_rank'])
    .drop_duplicates('gene_id')['transcript_id']
)

# Define start and stop codons
codons = df[
    (df['feature'].isin(['start_codon', 'stop_codon'])) &
    (df['transcript_id'].isin(final_best_ids))
].copy()
codons['interval'] = codons.apply(format_interval, axis=1)

codon_bounds = codons.groupby(['transcript_id', 'feature']).agg({'start': 'min', 'end': 'max'}).unstack()

# Classify UTR
feature_rows = df[
    (df['seqname'] != 'chrM') &
    (df['feature'].isin(['UTR', 'CDS'])) &
    (df['transcript_id'].isin(final_best_ids))
].copy()

feature_rows['feature'] = feature_rows.apply(lambda r: classify_utr(r, codon_bounds), axis=1)
feature_rows['interval'] = feature_rows.apply(format_interval, axis=1)

# Build coords including codons; merge codon coordinates into CDS but keep codon columns
final_features = ["5' UTR", 'start_codon', 'CDS', 'stop_codon', "3' UTR", 'appris_rank']
feature_table = feature_rows[feature_rows['feature'].isin(["5' UTR", 'CDS', "3' UTR"])][['gene_name', 'gene_id', 'transcript_id', 'feature', 'interval']]
codon_table = codons[['gene_name', 'gene_id', 'protein_id','transcript_id', 'feature', 'interval']]

initialDB_coords_df = (
    pd.concat([feature_table, codon_table], ignore_index=True)
    .groupby(['gene_name', 'gene_id', 'transcript_id', 'feature'], sort=False)['interval']
    .agg('; '.join)
    .unstack(fill_value='')
    .reindex(columns=final_features)
    .reset_index()
)

appris_map = transcripts.set_index('transcript_id')['appris_rank']
initialDB_coords_df['appris_rank'] = initialDB_coords_df['transcript_id'].map(appris_map).fillna(10).astype(int)

protein_map = (
    codons.dropna(subset=['protein_id'])
    .drop_duplicates('transcript_id')
    .set_index('transcript_id')['protein_id']
)
initialDB_coords_df['protein_id'] = initialDB_coords_df['transcript_id'].map(protein_map)
initialDB_coords_df = initialDB_coords_df[initialDB_coords_df['protein_id'].notna()].copy()

initialDB_coords_df['CDS'] = initialDB_coords_df.apply(merge_cds_with_codons_start_cds_stop, axis=1)
print(f"Initial coords table has {len(initialDB_coords_df)} rows before dropping unfinished transcripts")

# Drop transcripts with unfinished CDS or mRNA after coords are built
coords_drop_unfinished = (
    initialDB_coords_df['transcript_id'].isin(df.loc[df['unfinished_CDS'] | df['unfinished_mRNA'], 'transcript_id'])
)

rows_dropped = int(coords_drop_unfinished.sum())
initialDB_coords_df = initialDB_coords_df.loc[~coords_drop_unfinished].copy()
print(f"Dropped {rows_dropped} rows with unfinished CDS or mRNA tags")

Initial coords table has 20361 rows before dropping unfinished transcripts
Dropped 304 rows with unfinished CDS or mRNA tags


In [6]:
mt = initialDB_coords_df[initialDB_coords_df['gene_name'].str.contains('MT')]

initialDB_coords_df.drop(initialDB_coords_df[initialDB_coords_df['gene_name'].str.contains('MT')].index, inplace=True)

len(mt)

258

In [7]:
initialDB_coords_df.to_csv('../data/initialDB_coords.csv', index=False)
len(initialDB_coords_df) 

19799

In [8]:
assembly_path = Path('..') / 'data' / 'GRCh38.primary_assembly.genome.fa'

def load_genome_fasta(path):
    genome = {}
    with open(path) as f:
        chrom = None
        seq_chunks = []
        for line in f:
            if line.startswith('>'):
                if chrom: genome[chrom] = ''.join(seq_chunks)
                chrom = line[1:].split()[0]
                seq_chunks = []
            else:
                seq_chunks.append(line.strip())
        if chrom: genome[chrom] = ''.join(seq_chunks)
    return genome

genome = load_genome_fasta(assembly_path)

In [9]:
interval_pattern = re.compile(r'^(?P<chrom>[^:]+):(?P<start>\d+)-(?P<end>\d+)\((?P<strand>[+-])\)$')


def reverse_complement(seq):
    """Return the reverse complement of a DNA sequence."""
    return seq.translate(str.maketrans('ACGTacgtNn', 'TGCAtgcaNn'))[::-1]


def get_chrom_sequence(chrom):
    """Return the sequence for a chromosome, trying both 'chr' and non-'chr' versions."""
    if chrom in genome:
        return genome[chrom]
    alt_chrom = chrom[3:] if chrom.startswith('chr') else f'chr{chrom}'
    if alt_chrom in genome:
        return genome[alt_chrom]
    raise KeyError(f'Chromosome {chrom!r} not found in loaded genome')


def interval_to_sequence(interval):
    """Return the sequence for a given interval."""
    match = interval_pattern.match(interval)
    if match is None:
        raise ValueError(f'Could not parse interval {interval!r}')

    chrom = match.group('chrom')
    start = int(match.group('start'))
    end = int(match.group('end'))
    strand = match.group('strand')

    sequence = get_chrom_sequence(chrom)[start - 1:end]
    if strand == '-':
        sequence = reverse_complement(sequence)
    return sequence


def intervals_to_sequence(interval_string):
    """Return the sequences of intervals indicated by the coordinates. The coordinates are concatenated as the separation
    before was only due to the fact they were on different exons.
    """
    if pd.isna(interval_string) or interval_string == '':
        return ''
    return ''.join(
        interval_to_sequence(interval.strip())
        for interval in interval_string.split(';')
        if interval.strip()
    )


sequence_cols = ["5' UTR", 'start_codon', 'CDS', 'stop_codon', "3' UTR"]
initialDB_df = initialDB_coords_df.copy()
for col in sequence_cols:
    initialDB_df[col] = initialDB_df[col].apply(intervals_to_sequence)

# Ensure empty entries become empty string
for col in sequence_cols:
    initialDB_df[col] = initialDB_df[col].fillna('')

# Add length columns only for UTR/CDS (not start/stop codons)
length_cols = ["5' UTR", 'CDS', "3' UTR"]
for col in length_cols:
    initialDB_df[f"{col}_len"] = initialDB_df[col].apply(len)

# initialDB_df.to_csv('../data/initialDB_sequences.csv', index=False)
initialDB_df.head()

df = initialDB_df.copy()

In [10]:
not3 = initialDB_df[initialDB_df['CDS_len'] % 3 != 0].copy().drop(columns=["5' UTR", "3' UTR", "5' UTR_len", "3' UTR_len"])

In [11]:
initialDB_df = initialDB_df[(initialDB_df['5\' UTR_len'] >= 10) & (initialDB_df['CDS_len'] >= 150)]

len(initialDB_df)

18995

In [12]:
# Protein Sequence 
codons = {
"TTT" : "F", "CTT" : "L", "ATT" : "I", "GTT" : "V", "TTC" : "F",
"CTC" : "L", "ATC" : "I", "GTC" : "V", "TTA" : "L", "CTA" : "L",
"ATA" : "I", "GTA" : "V", "TTG" : "L", "CTG" : "L", "ATG" : "M",
"GTG" : "V", "TCT" : "S", "CCT" : "P", "ACT" : "T", "GCT" : "A",
"TCC" : "S", "CCC" : "P", "ACC" : "T", "GCC" : "A", "TCA" : "S",
"CCA" : "P", "ACA" : "T", "GCA" : "A", "TCG" : "S", "CCG" : "P",
"ACG" : "T", "GCG" : "A", "TAT" : "Y", "CAT" : "H", "AAT" : "N",
"GAT" : "D", "TAC" : "Y", "CAC" : "H", "AAC" : "N", "GAC" : "D",
"TAA" : "Stop", "CAA" : "Q", "AAA" : "K", "GAA" : "E", "TAG" : "Stop", 
"CAG" : "Q", "AAG" : "K", "GAG" : "E", "TGT" : "C", "CGT" : "R",
"AGT" : "S", "GGT" : "G", "TGC" : "C", "CGC" : "R", "AGC" : "S",
"GGC" : "G", "TGA" : "Stop", "CGA" : "R", "AGA" : "R", "GGA" : "G",
"TGG" : "W", "CGG" : "R", "AGG" : "R", "GGG" : "G"
}

def translate_cds_to_protein(cds_seq):
    if pd.isna(cds_seq) or len(cds_seq) < 3:
        return np.nan
    
    cds_seq = cds_seq.upper()
    protein_seq = ''
    
    for i in range(0, len(cds_seq) - 2, 3):
        codon = cds_seq[i:i+3]
        amino_acid = codons.get(codon, 'X')  # 'X' for unknown codons
        if amino_acid == "Stop":
            break
        protein_seq += amino_acid
    
    return protein_seq

initialDB_df['protein_sequence'] = initialDB_df['CDS'].apply(translate_cds_to_protein)

In [13]:
# Load gencode protein translations keyed by transcript ID
from pathlib import Path

gencode_fasta_path = Path('..') / 'data' / 'gencode.v49.pc_translations.fa'

def load_protein_fasta_by_transcript(fasta_path):
    """Load FASTA and map transcript ID to amino acid sequence."""
    protein_dict = {}
    with open(fasta_path) as f:
        current_id = None
        current_seq = []
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if current_id is not None:
                    protein_dict[current_id] = ''.join(current_seq)
                header_parts = line[1:].split('|')
                current_id = header_parts[1] if len(header_parts) > 1 else header_parts[0]
                current_seq = []
            else:
                current_seq.append(line)
        if current_id is not None:
            protein_dict[current_id] = ''.join(current_seq)
    return protein_dict

# load by transcript ID
mappings_by_transcript = load_protein_fasta_by_transcript(gencode_fasta_path)
print(f"Loaded {len(mappings_by_transcript)} transcript AA sequences from gencode")

# Compare protein_sequence directly against the gencode sequence for the same transcript ID

def matches_gencode_by_transcript(row, gencode_by_transcript):
    """Return True if the translated protein sequence equals the gencode translation for the transcript."""
    aa_seq = row['protein_sequence']
    tid = row['transcript_id']
    if pd.isna(aa_seq) or aa_seq == '':
        return False
    return aa_seq == gencode_by_transcript.get(tid, '')

# Apply direct transcript-ID based matching
initialDB_df['matches_gencode_aa'] = initialDB_df.apply(lambda r: matches_gencode_by_transcript(r, mappings_by_transcript), axis=1)
initialDB_df.drop

print(f"\nMatches found: {initialDB_df['matches_gencode_aa'].sum()}/{len(initialDB_df)}")

initialDB_df = initialDB_df[(initialDB_df['matches_gencode_aa'] == True)]

Loaded 245535 transcript AA sequences from gencode

Matches found: 18912/18995


In [14]:
# For now, drop all sequences in not3 -> revisit later if needed

initialDB_df = initialDB_df[initialDB_df['CDS_len'] % 3 == 0].copy()
initialDB_df.to_csv('../data/initialDB_sequences.csv', index=False)

len(initialDB_df)

18906

In [15]:
initialDB_df.head()

feature,gene_name,gene_id,transcript_id,5' UTR,start_codon,CDS,stop_codon,3' UTR,appris_rank,protein_id,5' UTR_len,CDS_len,3' UTR_len,protein_sequence,matches_gencode_aa
0,OR4F5,ENSG00000186092.7,ENST00000641515.2,CCCAGATCTCTTCAGTTTTTATGCCTCATTCTGTGAAAATTGCTGT...,ATG,ATGAAGAAGGTAACTGCAGAGGCTATTTCCTGGAATGAATCAACGA...,TAG,,1,ENSP00000493376.2,60,981,0,MKKVTAEAISWNESTSETNNSMVTEFIFLGLSDSQELQTFLFMLFF...,True
3,SAMD11,ENSG00000187634.14,ENST00000616016.5,GGCGGCGGAGTCTCCCAAGTCCCCGCCGGGCGGGCGCGCGCCAGTG...,ATG,ATGCCGGCGGTCAAGAAGGAGTTCCCGGGCCGCGAGGACCTGGCCC...,TGA,,10,ENSP00000478421.2,509,2535,0,MPAVKKEFPGREDLALALATFHPTLAALPLPPLPGYLAPLPAAAAL...,True
4,NOC2L,ENSG00000188976.12,ENST00000327044.7,GCTTCGGGTTGGTGTC,ATG,ATGGCAGCTGCGGGGAGCCGCAAGAGGCGCCTGGCGGAGCTGACGG...,TGA,TGAGGCAGCCCATCTGGGGGGCCTGTAGGGGCTGCCGGGCTGGTGG...,3,ENSP00000317992.6,16,2250,494,MAAAGSRKRRLAELTVDEFLASGFDSESESESENSPQAETREAREA...,True
5,KLHL17,ENSG00000187961.16,ENST00000338591.8,GGGAGTGAGCGACACAGAGCGGGCCGCCACCGCCGAGCAGCCCTCC...,ATG,ATGCAGCCCCGCAGCGAGCGCCCGGCCGGCAGGACGCAGAGCCCGG...,TGA,,1,ENSP00000343930.3,110,1929,0,MQPRSERPAGRTQSPEHGSPGPGPEAPPPPPPQPPAPEAERTRPRQ...,True
6,PLEKHN1,ENSG00000187583.12,ENST00000379410.8,AGGAGGCTGTGGACAGGGACCCAGACTTGCCGACCTGTACGACTCT...,ATG,ATGGGGAACAGCCACTGTGTCCCTCAGGCCCCCAGGAGGCTCCGGG...,TGA,,2,ENSP00000368720.3,50,1836,0,MGNSHCVPQAPRRLRASFSRKPSLKGNREDSARMSAGLPGPEAARS...,True
